# M3：通勤成本变化

<div class="page-intro">

上一章已经建立居住地与工作地的联合选择，本章沿用这一机制，把政策从住房扩建改为双向跨区通勤成本下降。为了看清就业中心与居住区的不同调整，我们改用非对称基准，并在政策计算中固定已校准的住房、生产率和便利度。先固定工资与房租看选择如何变化，再让价格参与调整，比较居民、就业、通勤与工人事前福利的均衡变化。

</div>

**两种实践方式：** 展开绿色 Python 代码，理解公式与算法；使用浅紫色的讲义练习完成当前任务，或用浅蓝色的迁移模板描述自己的研究。两类卡片均默认收起。开始前请读[提示词使用指南](prompts/usage-guide.md)，按卡片准备材料；迁移模板中的 `{……}` 需要替换。

::: {.model-path}

[M1 无通勤](https://lianxhcn.github.io/qsm101/site/pages/m1-baseline.html) → [M2 加入通勤](https://lianxhcn.github.io/qsm101/site/pages/m2-commuting.html) → **M3 通勤成本变化 (本章)**

:::

[下载本章 Notebook](transport.ipynb) · [下载基准数据](data/teaching-central-residential-baseline.csv) · [查看模型设定](model.md) · [下载离线练习包](https://lianxhcn.github.io/qsm101/articles/p06-transport-qsm/downloads/transport-workbook.zip)

::: {.callout-note title="开始之前" icon=false}

建议先读 [M2：加入跨地区通勤](https://lianxhcn.github.io/qsm101/site/pages/m2-commuting.html)，理解四种居住地与工作地组合。这里直接设定行为参数，不新增模拟样本估计环节。

:::

完成本章后，你应能：

- 说明非对称基准怎样产生就业中心与居住区的分工。
- 在同一基本面下比较固定价格预测与一般均衡。
- 分别解释居民、就业、双向通勤与工人事前福利的变化。
- 识别对称性与参数设定对结论的影响。

::: {.callout-tip title="离线运行" icon=false}

Notebook 包含全部模型函数，可独立运行，不调用仓库中的参考脚本。解压离线练习包后，保留 Notebook 与 `data/` 的相对位置；在已有 Python 环境中执行 `python -X utf8 -m pip install -r requirements.txt`，再用 Jupyter 打开并从头运行。网页保留结果，便于先读模型再下载练习。

:::



## 1. 为什么要换一个基准

住在 B、到 A 上班的人，可能因通勤方便而继续住在 B；住在 B、也在 B 工作的人，则可能转向 A 的岗位。居住与就业因此可能向不同地区调整。

前一章从完全对称的两地出发：住房存量、生产率和居住便利度相同，双向通勤成本也相同。若仍从这个基准出发，并将两个方向的成本同幅降低，两地的居民数、就业数、工资和房租都保持原值，只有双向通勤增加。

这并不表示交通改善没有作用。对称设定没有提供一个让就业或居住偏向某地的理由。要讨论就业中心与居住区的分工，本章让 A 的基准工资较高、B 的基准房租较低，同时保留相同的双向跨区成本。

![从无通勤基准、对称通勤基准到非对称交通反事实。教学示意。](figures/qsm-transport-fig01-model-evolution-20261006-221849.png){#fig-m3-transport-policy-model-evolution fig-alt="从无通勤基准、对称通勤基准到非对称交通反事实" .dense-figure}


@fig-m3-transport-policy-model-evolution 的中栏对应完全对称的教学基准；右栏对应本章的非对称基准：A 的生产率和工资较高，B 的住房较多、房租较低，基准中 B → A 的通勤人数更多。右栏较粗的箭头表示基准流量较大，没有预设政策变化的幅度。两栏的双向跨区成本都相同，非对称性来自地区基本面。住房反馈框作了简写，房租还取决于当地居民在各工作地取得的工资。

仍用 $M_{ij}$ 表示住在 $i$、工作在 $j$ 的工人数。只需记住前章的两个合计：

$$
R_{i}=\sum_{j}M_{ij},\qquad E_{j}=\sum_{i}M_{ij}.
$$ {#eq-m3-transport-policy-accounting}

$R_{i}$ 按居住地统计，$E_{j}$ 按工作地统计。这里的居民只包括模型工人。B → A 的通勤增加，关联的是 B 的住房需求与 A 的劳动投入。后面每张表都按这一方向记账。

## 2. 怎样构造就业中心与居住区

先给定两地的居民数、工资和租金。租金是每平方米每月的价格，居民数是模型工人数。以下三列是原始输入，后面的就业和通勤流量由模型构造。

In [1]:
# 准备计算工具。路径同时兼容仓库根目录、网页源目录和解压后的练习包。
from pathlib import Path
from datetime import datetime, timezone
import copy, hashlib, importlib.metadata, json, platform, sys
import numpy as np
import pandas as pd
from scipy.optimize import root
from scipy.special import softmax, logsumexp
from IPython.display import display, HTML

TAU0 = np.log(9.) / 4.
STARTS = [(0.,0.), (-2.,-2.), (2.,2.), (-2.,2.), (2.,-2.)]
REGIONS = ['A','B']
NATURE = '教学构造；非真实城市或交通项目评估'
candidates = [Path.cwd(), *Path.cwd().parents]
DATA = next((folder / suffix for folder in candidates
             for suffix in ['data', 'articles/p06-transport-qsm/data']
             if (folder / suffix / 'teaching-central-residential-primitives.csv').exists()), None)
if DATA is None:
    raise FileNotFoundError('请先解压完整练习包，或从 qsm101 仓库中运行。')
output = Path.cwd() / 'runs' / ('p06-notebook-' + datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
output.mkdir(parents=True, exist_ok=False)

In [2]:
def read_inputs(data):
    """严格检查标签与四格方向，绝不从显示表的圆整数读取。"""
    names = ['teaching-central-residential-primitives.csv', 'teaching-central-residential-od.csv']
    hashes = {name: hashlib.sha256((data/name).read_bytes()).hexdigest() for name in names}
    df = pd.read_csv(data/names[0]).set_index('region').loc[REGIONS]
    od = pd.read_csv(data/names[1])
    if len(df) != 2 or len(od) != 4 or od.duplicated(['home_region','work_region']).any():
        raise ValueError('需要两个地区和四个不重复的 OD 单元')
    M = od.pivot(index='home_region', columns='work_region', values='workers').loc[REGIONS, REGIONS].to_numpy()
    arr = df[['residents','monthly_wage','monthly_rent_per_m2']].to_numpy()
    if not np.isfinite(arr).all() or (arr <= 0).any() or not np.isfinite(M).all() or (M <= 0).any():
        raise ValueError('固定输入必须是有限正数')
    return arr[:,0], arr[:,1], arr[:,2], M, hashes

In [3]:
# 完整精度输入用于计算；只在展示时保留两位小数。
R, w, r, M_input, hashes = read_inputs(DATA)
primitive_table = pd.DataFrame({'居民数 (人)': R, '月工资 (元/工人)': w,
                                '月租金 (元/平方米)': r}, index=REGIONS)
display(primitive_table.style.format('{:,.2f}'))

,居民数 (人),月工资 (元/工人),月租金 (元/平方米)
A,"50,000.00","10,000.00",140.00
B,"70,000.00","8,200.00",100.00


沿用住房支出份额 $\alpha=0.25$、生产弹性 $\beta=0.90$ 和选择参数 $\theta=4$。总工人数为 $N=120000$。本地通勤成本为零，双向跨区成本均为 $\tau_{0}=\ln(9)/4$。

给定居住地后，先比较两个工作地的工资与通勤不便。条件工作地概率为：

$$
q_{j\mid i}=\frac{\exp\{\theta(\ln w_{j}-\tau_{ij})\}}
{\sum_{\ell}\exp\{\theta(\ln w_{\ell}-\tau_{i\ell})\}},
\qquad M_{ij}^{0}=R_{i}^{0}q_{j\mid i}^{0}.
$$ {#eq-m3-transport-policy-conditional-choice}

同一居住地的房租和便利度在两个工作地之间相同，因此在条件概率中消去。将每行的概率乘以该地居民数，就得到通勤表；对列求和，才得到就业数。

In [4]:
def calibrate(R, w, r, theta=4., beta=.9, tau=TAU0):
    """给定原始 R/w/r/tau。theta 改变后重建 OD，不强行匹配旧 OD。"""
    if theta <= 0 or not 0 < beta <= 1 or tau < 0:
        raise ValueError('theta>0，0<beta<=1，tau>=0')
    R, w, r = map(lambda x: np.asarray(x, dtype=float), (R,w,r))
    alpha = .25
    cost = np.array([[0.,tau],[tau,0.]])
    scores = theta*(np.log(w)[None,:]-cost)
    q = softmax(scores, axis=1)
    M = R[:,None]*q
    E = M.sum(0)
    H = alpha*(M@w)/r
    Z = w*E**(1-beta)/beta
    inclusive = logsumexp(scores, axis=1)
    logB = (np.log(R/R[0])+alpha*theta*np.log(r/r[0])-(inclusive-inclusive[0]))/theta
    p = dict(N=float(R.sum()), alpha=alpha, beta=beta, theta=theta,
             H=H, Z=Z, B=np.exp(logB), tau=cost)
    target = dict(R=R,E=E,w=w,r=r,M=M,H=H,Z=Z,B=p['B'])
    return p, target

In [5]:
# 重建基准 OD；行表示居住地，列表示工作地。
p, target = calibrate(R, w, r)
q0 = target['M'] / R[:, None]
display(pd.DataFrame(q0, index=['住 A','住 B'], columns=['在 A 工作','在 B 工作']).style.format('{:.2%}'))
od0 = pd.DataFrame(target['M'], index=['住 A','住 B'], columns=['在 A 工作','在 B 工作'])
od0['居民合计'] = R
od0.loc['就业合计'] = [*target['E'], p['N']]
display(od0.style.format('{:,.2f}'))

,在 A 工作,在 B 工作
住 A,95.22%,4.78%
住 B,19.73%,80.27%


,在 A 工作,在 B 工作,居民合计
住 A,"47,608.36","2,391.64","50,000.00"
住 B,"13,809.17","56,190.83","70,000.00"
就业合计,"61,417.53","58,582.47","120,000.00"


看第二行：B 的 70,000 名居民中，约 19.73% 到 A 工作。看第一列：A 吸收了约 61,417.53 个岗位，高于其居民数。A 的工资更高，但通勤成本仍使多数 B 居民留在 B 工作。

接下来要找一组基本面，使这张表与给定的工资、租金共同构成基准均衡。用 $H_{i}$ 表示有效住房服务存量，$Z_{j}$ 表示包含固定生产要素的生产率项。由住房出清和工资等于劳动边际产出，反演得到：

$$
H_{i}=\frac{\alpha\sum_{j}w_{j}^{0}M_{ij}^{0}}{r_{i}^{0}},
\qquad
Z_{j}=\frac{w_{j}^{0}(E_{j}^{0})^{1-\beta}}{\beta}.
$$ {#eq-m3-transport-policy-fundamentals}

住房反演使用住在当地的全部工人工资，包括他们在另一地区挣到的钱。便利度 $B_{i}$ 则用来匹配居民在两地的分配；令 $B_{A}=1$ 确定尺度，再求出 $B_{B}$。完整公式见[模型说明](model.md)。

<details class="agent-prompt" id="agent-m3-calibration" data-prompt-kind="exercise">
<summary><span class="agent-prompt-toggle" aria-hidden="true"></span><span>Agent · 讲义练习：构造非对称基准，先核对口径</span></summary>
<div class="agent-prompt-body">
<div data-prompt-role="materials">

**准备材料：** 下载并交给 Agent：[模型说明](model.md) · [teaching-central-residential-baseline.csv](data/teaching-central-residential-baseline.csv)。 实现或复查时另提供已有程序、校准结果和本次输出；可用[教学 Notebook](transport.ipynb)作参考。

</div>
<div data-prompt-role="scope">

**适用条件：** 给定行为参数和相同双向通勤成本，按原始居民与价格重建非对称基准，不新增估计步骤。

</div>
<div data-prompt-role="task">

**练习任务**

```text
我提供了非对称两地区通勤成本政策模型的设定与此任务所需的输入材料。
请阅读所提供的模型说明文件与 data/ 下的原始输入，区分居民、工资、租金与由模型生成的就业和 OD。
采用模型说明给定的 alpha、beta、theta、tau0，不新增参数估计环节。
先由工资和通勤成本计算条件工作地概率，再用居民数重建四格 OD 与就业。
按住房支出反演 H、按劳动边际产出反演 Z，并按 B_A=1 反演相对便利度。
先写出各步骤的方程依赖，再编写或核查代码并实际运行，报告基准复原误差。
所提供模型的非对称性来自地区基本面，不要擅自设置两个方向不同的通勤成本。
若材料缺失，请列明缺口；需要运行却无法运行时，区分拟执行步骤与已完成结果。
```

</div>
<div data-prompt-role="checks">

**拿到回复后看什么：** 原始输入与模型反演是否分开；是否仍使用本章相同的双向成本。

</div>
</div>
</details>



In [6]:
# 校准基本面与原始输入分开显示，避免把反演值当成观测值。
fundamentals = pd.DataFrame({'有效住房服务 H (平方米)': p['H'],
                             '生产率项 Z': p['Z'], '居住便利度 B': p['B']}, index=REGIONS)
display(fundamentals.style.format('{:,.4f}'))

,有效住房服务 H (平方米),生产率项 Z,居住便利度 B
A,"885,169.7233","33,464.6836",1.0000
B,"1,497,141.2595","27,311.6610",1.1686


这组反演结果让 A 具有较高的生产率，让 B 具有较多的住房和较高的居住便利度。它服务于本章的中心—居住区算例。基准能被复原，说明方程、输入和代码相容，不能据此声称已识别真实城市的生产率或便利度。

政策计算从这里开始固定 $H,Z,B,N,\alpha,\beta,\theta$，只降低双向跨区成本。若在政策后重新校准基本面，就无法区分交通改善与基本面变化各自的作用。

## 3. 先固定价格，通勤选择会怎样变

先暂时固定工资和租金，只改变跨区成本：

$$
\tau_{AB}^{1}=\tau_{BA}^{1}=0.75\tau_{0},\qquad
\tau_{AA}^{1}=\tau_{BB}^{1}=0.
$$ {#eq-m3-transport-policy-policy}

这里的 25% 是效用单位中的广义通勤不便降幅，不能直接解释为真实通勤时间或票价下降 25%。

对住在 B 的工人而言，在 A 工作相对于在 B 工作的概率比为：

$$
\frac{q_{A\mid B}}{q_{B\mid B}}
=\left(\frac{w_{A}}{w_{B}}\right)^{\theta}e^{-\theta\tau_{BA}}.
$$ {#eq-m3-transport-policy-choice-odds}

工资不变时，成本降低使这个比值乘以 $e^{\theta(0.25\tau_{0})}=9^{1/4}\approx1.732$。这说明 A 的岗位相对更有吸引力，但不意味着 A 的就业或 B → A 的人数增加 73.2%。概率还需归一化，居住人数也会变化。

把居住选择一起放回来。沿用前章的系统效用与联合概率：

$$
V_{ij}=\ln B_{i}+\ln w_{j}-\alpha\ln r_{i}-\tau_{ij},\qquad
p_{ij}=\frac{e^{\theta V_{ij}}}{\sum_{k,\ell}e^{\theta V_{k\ell}}},\qquad M_{ij}=Np_{ij}.
$$ {#eq-m3-transport-policy-joint-choice}

这一 Logit 形式来自独立、尺度为 $1/\theta$ 的 I 型极值偏好冲击。下面仅更新成本，然后重算四种组合的概率。

<details class="agent-prompt" id="agent-m3-fixed" data-prompt-kind="exercise">
<summary><span class="agent-prompt-toggle" aria-hidden="true"></span><span>Agent · 讲义练习：只看通勤变化，暂时固定价格</span></summary>
<div class="agent-prompt-body">
<div data-prompt-role="materials">

**准备材料：** 下载并交给 Agent：[模型说明](model.md) · [teaching-central-residential-baseline.csv](data/teaching-central-residential-baseline.csv)。 实现或复查时另提供已有程序、校准结果和本次输出；可用[教学 Notebook](transport.ipynb)作参考。

</div>
<div data-prompt-role="scope">

**适用条件：** 只计算价格固定时的直接响应，不要求工资和住房市场重新出清；广义成本不等同票价。

</div>
<div data-prompt-role="task">

**练习任务**

```text
我提供了非对称两地区通勤成本政策模型的设定与此任务所需的输入材料。
请在非对称两地区通勤政策模型的基准工资与租金保持不变时，把双向跨区 tau 同时乘以 0.75，区内成本保持为零。
固定 H、Z、B、N 和行为参数，用联合选择概率重新计算 M、R、E。
先说明计算步骤，再实际运行并输出各地区居民、就业和四格流量。
将结果明确标为固定价格预测；检查它与工资边际产出、住房出清之间的偏离。
不要为使市场出清而偷偷更新价格，也不要将该预测称为一般均衡。
解释这是效用单位的广义成本变化，不直接等同于票价或通勤时间下降 25%。
若材料缺失，请列明缺口；需要运行却无法运行时，区分拟执行步骤与已完成结果。
```

</div>
<div data-prompt-role="checks">

**拿到回复后看什么：** 工资与租金是否确实固定；为何新的分布还需要市场反馈。

</div>
</div>
</details>



In [7]:
def changed(p, fraction):
    """只改变 tau，深拷贝以避免污染基准基本面。"""
    out = copy.deepcopy(p)
    out['tau'] *= 1-fraction
    return out

In [8]:
# 固定价格的选择预测：居住与工作都可变，工资和租金暂不调整。
p1 = changed(p, .25)
v = np.log(p['B'])[:,None] + np.log(w)[None,:] - p['alpha']*np.log(r)[:,None] - p1['tau']
fixed_M = p['N'] * softmax(p['theta'] * v)
fixed = dict(M=fixed_M, R=fixed_M.sum(1), E=fixed_M.sum(0), w=w, r=r,
             logW=logsumexp(p['theta']*v)/p['theta'])
fixed_table = pd.DataFrame({'基准居民':R, '固定价格预测居民':fixed['R'],
                            '基准就业':target['E'], '固定价格预测就业':fixed['E']}, index=REGIONS)
display(fixed_table.style.format('{:,.2f}'))

,基准居民,固定价格预测居民,基准就业,固定价格预测就业
A,"50,000.00","47,096.20","61,417.53","65,093.26"
B,"70,000.00","72,903.80","58,582.47","54,906.74"


固定价格预测中，B 居民增加到约 72,903.80 人，A 就业增加到约 65,093.26 人。较低房租与较高工资更容易通过跨区组合连接起来，居住和就业向不同地区集中。

这个预测尚未满足新的市场出清条件。A 的岗位增加会改变劳动边际产出；B 的居民增加，以及更多居民在 A 取得工资，会改变住房支出。因此，必须让工资和房租随之调整，再计算选择。

## 4. 让工资和房租参与调整

工人只有工资收入，预算为 $c+r_{i}h=w_{j}$，住房支出占工资的 $\alpha$。住房和固定生产要素由区外所有者持有。给定基本面，两个市场分别要求：

$$
Y_{j}=Z_{j}E_{j}^{\beta},\qquad
w_{j}=\beta Z_{j}E_{j}^{\beta-1},\qquad
r_{i}H_{i}=\alpha\sum_{j}w_{j}M_{ij}.
$$ {#eq-m3-transport-policy-markets}

当 $\beta<1$ 时，A 就业增加会压低 A 的工资，削弱继续向 A 集中的动力。B 的住房存量固定，居民工资性住房支出增加会抬高 B 的房租，削弱继续住在 B 的吸引力。B 的住房支出不能写成 $\alpha w_{B}R_{B}$，因为部分 B 居民在 A 取得工资。

仍可用前章的两个未知量求解：$x=\ln(R_{B}/R_{A})$、$y=\ln(E_{B}/E_{A})$。给定候选居民数与就业数，先算工资，再用条件工作地概率算居民平均工资及候选房租，随后重算联合通勤表。当新表的行列边际与候选值一致时，市场闭合。下面先定义这一映射，再用它分别求基准和政策均衡。

<details class="agent-prompt" id="agent-m3-equilibrium" data-prompt-kind="exercise">
<summary><span class="agent-prompt-toggle" aria-hidden="true"></span><span>Agent · 讲义练习：让价格反馈进入同一个反事实</span></summary>
<div class="agent-prompt-body">
<div data-prompt-role="materials">

**准备材料：** 下载并交给 Agent：[模型说明](model.md) · [teaching-central-residential-baseline.csv](data/teaching-central-residential-baseline.csv)。 实现或复查时另提供已有程序、校准结果和本次输出；可用[教学 Notebook](transport.ipynb)作参考。

</div>
<div data-prompt-role="scope">

**适用条件：** 在已校准非对称基准上只改变跨区广义成本，政策内不重新校准基本面。

</div>
<div data-prompt-role="task">

**练习任务**

```text
我提供了非对称两地区通勤成本政策模型的设定与此任务所需的输入材料。
请在非对称两地区通勤政策模型的同一基准上求双向通勤成本下降 25% 的一般均衡。
固定已校准的 H、Z、B、N、alpha、beta、theta，仅改变跨区 tau；政策后不重新校准基本面。
以居民和就业 B/A 对数比为未知量，让工资、租金与联合选择共同满足市场条件。
住房支出必须汇总居民在各工作地取得的工资，不能用本地工资乘当地居民数替代。
先复原基准，再实际运行政策情景；并列输出基准、固定价格预测与均衡结果。
分别报告居民、就业、四格通勤、净通勤、总通勤与事前福利指数。
保留求解状态和经济残差，用价格反馈解释两类预测的差异，不预设所有变化都被削弱。
若材料缺失，请列明缺口；需要运行却无法运行时，区分拟执行步骤与已完成结果。
```

</div>
<div data-prompt-role="checks">

**拿到回复后看什么：** 政策内基本面是否固定；两种预测是否共享同一基准与冲击。

</div>
</div>
</details>



In [9]:
def state(xy, p, restricted=False):
    """以候选 R/E 构造价格，再重算联合流量和对数边际比残差。"""
    R = p['N']*softmax([0.,xy[0]])
    E = p['N']*softmax([0.,xy[1]])
    w = p['beta']*p['Z']*E**(p['beta']-1)
    mask = np.eye(2,dtype=bool) if restricted else np.ones((2,2),dtype=bool)
    score = np.where(mask,p['theta']*(np.log(w)[None,:]-p['tau']),-np.inf)
    q = softmax(score,axis=1)
    r = p['alpha']*R*(q@w)/p['H']
    v = np.log(p['B'])[:,None]+np.log(w)[None,:]-p['alpha']*np.log(r)[:,None]-p['tau']
    joint_score = np.where(mask,p['theta']*v,-np.inf)
    prob = np.exp(joint_score-logsumexp(joint_score))
    M = p['N']*prob
    rows, cols = M.sum(1), M.sum(0)
    residual = np.log([rows[1]/rows[0],cols[1]/cols[0]])-xy
    return dict(R=R,E=E,w=w,r=r,M=M,prob=prob,logW=logsumexp(joint_score)/p['theta'],
                residual=residual,xy=np.asarray(xy),Y=p['Z']*E**p['beta'])

In [10]:
def solve(p, start=(0.,0.), diagnostics=None, name='', restricted=False):
    """记录所有算法状态；最终验收还须检查经济方程。"""
    for method in ['hybr','lm']:
        sol = root(lambda xy: state(xy,p,restricted)['residual'],start,method=method,tol=1e-11)
        s = state(sol.x,p,restricted)
        err = float(np.max(np.abs(s['residual'])))
        if diagnostics is not None:
            diagnostics.append(dict(scenario=name,method=method,start=list(start),success=bool(sol.success),
                                    message=str(sol.message),nfev=int(sol.nfev),max_residual=err,solution=sol.x.tolist()))
        if err <= 1e-10: return s
    raise RuntimeError(f'{name}: 二维残差 {err} 未通过；不生成政策结论')

In [11]:
# 先复原基准，再在相同基本面下求政策均衡；记录每次求解器状态。
diagnostics, checks = [], []
b = solve(p, diagnostics=diagnostics, name='baseline')
s = solve(p1, diagnostics=diagnostics, name='policy25')
labels = {'R':'居民数 (人)', 'E':'就业数 (人)', 'w':'月工资 (元/工人)', 'r':'月租金 (元/平方米)'}
main_rows = [dict(变量=labels[key], 地区=reg, 基准=b[key][i], 政策后=s[key][i],
                  变化率=100*(s[key][i]/b[key][i]-1))
             for key in labels for i,reg in enumerate(REGIONS)]
display(pd.DataFrame(main_rows).style.format({'基准':'{:,.2f}', '政策后':'{:,.2f}', '变化率':'{:+.2f}%'}))

,变量,地区,基准,政策后,变化率
0,居民数 (人),A,"50,000.00","48,572.12",-2.86%
1,居民数 (人),B,"70,000.00","71,427.88",+2.04%
2,就业数 (人),A,"61,417.53","65,100.80",+6.00%
3,就业数 (人),B,"58,582.47","54,899.20",-6.29%
4,月工资 (元/工人),A,"10,000.00","9,941.93",-0.58%
5,月工资 (元/工人),B,"8,200.00","8,253.42",+0.65%
6,月租金 (元/平方米),A,140.00,134.45,-3.97%
7,月租金 (元/平方米),B,100.00,104.25,+4.25%


B 居民增加约 2.04%，就业却减少约 6.29%；A 的居民减少，就业增加约 6.00%。在这个基准中，交通改善使居住更偏向 B，岗位更集中于 A。

工资与租金的变化符合市场反馈：A 就业增加，工资下降约 0.58%；B 就业减少，工资上升约 0.65%。B 的租金提高约 4.25%。这一租金变化同时反映居民人数与居民平均工资的变化，不能只用本地就业人数解释。

再把总量拆回四格通勤表，查看哪些组合发生变化。

In [12]:
# 展示四格流量，避免只看净流入而漏掉反方向的通勤。
od_rows = [dict(居住地到工作地=f'{home} → {job}', 基准=b['M'][i,j], 政策后=s['M'][i,j])
           for i,home in enumerate(REGIONS) for j,job in enumerate(REGIONS)]
display(pd.DataFrame(od_rows).style.format({'基准':'{:,.2f}', '政策后':'{:,.2f}'}))

,居住地到工作地,基准,政策后
0,A → A,"47,608.36","44,504.20"
1,A → B,"2,391.64","4,067.92"
2,B → A,"13,809.17","20,596.60"
3,B → B,"56,190.83","50,831.28"


双向跨区通勤都增加，但 B → A 增加更多。跨区通勤总量为 $M_{AB}+M_{BA}$，占全部工人的比例从 13.50% 升至 20.55%；B 到 A 的净通勤量为 $M_{BA}-M_{AB}$，从 11,417.53 增至 16,528.67 人。还可以用 $R_{B}-E_{B}=M_{BA}-M_{AB}$ 检查记账。

这些是政策前后两张均衡分布的比较，没有追踪每名工人的搬家或换岗过程。例如，$M_{BB}$ 减少不能直接解读为这些人全部转入 $M_{BA}$。

最后，将完整均衡与刚才的固定价格预测放在一起。

In [13]:
def summary(s, baseline):
    M=s['M']
    return dict(cross_workers=float(M[0,1]+M[1,0]), cross_share=float((M[0,1]+M[1,0])/M.sum()),
                net_B_to_A=float(M[1,0]-M[0,1]), welfare_pct=float(100*np.expm1(s['logW']-baseline['logW'])))

In [14]:
# 用同一基准比较直接选择与价格反馈后的结果。
comparison = []
for label, result in [('固定价格预测',fixed), ('一般均衡',s)]:
    metrics = summary(result,b)
    comparison.append({'情景':label, 'B 居民 (人)':result['R'][1], 'A 就业 (人)':result['E'][0],
                       '跨区通勤占比 (%)':100*metrics['cross_share']})
display(pd.DataFrame(comparison).style.format({key:'{:,.2f}' for key in comparison[0] if key!='情景'}))

,情景,B 居民 (人),A 就业 (人),跨区通勤占比 (%)
0,固定价格预测,"72,903.80","65,093.26",21.28
1,一般均衡,"71,427.88","65,100.80",20.55


价格调整明显收回了一部分 B 居民增量，但两个情景的 A 就业人数非常接近。各个市场的反馈会共同作用，不能据此概括为一般均衡总会按同一比例缩小所有变化。

工人的事前福利指数沿用联合选择模型的汇总形式：

$$
W=\left[\sum_{i,j}\left(\frac{B_{i}w_{j}}{r_{i}^{\alpha}}e^{-\tau_{ij}}\right)^{\theta}\right]^{1/\theta}.
$$ {#eq-m3-transport-policy-welfare}

程序计算 $100(W_{1}/W_{0}-1)$，得到约 2.14%。这是工人选择机会与通勤不便变化对应的事前福利增幅，不能解释为每名工人的工资提高 2.14%。本模型没有计入建设成本、交通部门收入、拥堵外部性及住房和固定要素所有者的福利，因而该数值也不等于交通项目的净社会福利。

In [15]:
print(f"工人事前福利指数变化：{summary(s,b)['welfare_pct']:.4f}%")

工人事前福利指数变化：2.1376%


## 5. 哪些结果依赖设定

可以先做一个对照：回到前章的完全对称基准，双向通勤成本同幅下降后，居民、就业和价格都保持不变，双向通勤仍增加。这说明本章的空间再分配需要地区差异提供方向。下面的补充核验还检查零冲击、高成本、本地工作限制、零成本下的概率可分离性，以及互换地区标签和更换求解初值。它们能发现程序或记账错误，不能证明模型在所有参数下只有一个均衡。

<details class="agent-prompt" id="agent-m3-verify" data-prompt-kind="exercise">
<summary><span class="agent-prompt-toggle" aria-hidden="true"></span><span>Agent · 讲义练习：用对称与极端情形检验结果</span></summary>
<div class="agent-prompt-body">
<div data-prompt-role="materials">

**准备材料：** 下载并交给 Agent：[模型说明](model.md) · [teaching-central-residential-baseline.csv](data/teaching-central-residential-baseline.csv)。 实现或复查时另提供已有程序、校准结果和本次输出；可用[教学 Notebook](transport.ipynb)作参考。

</div>
<div data-prompt-role="scope">

**适用条件：** 按非对称两地区通勤模型规定检查；对称性检验使用单独构造的对称基准。

</div>
<div data-prompt-role="task">

**练习任务**

```text
我提供了非对称两地区通勤成本政策模型的设定与此任务所需的输入材料。
请按所提供的非对称通勤模型说明文件重新核算经济条件，不只读取求解器的 success。
检查 M 的行列合计、总人口、选择概率、住房支出与工作地工资条件。
检查零冲击返回基准；对称基准下同幅降成本时，两地区总量不变而跨区通勤可增加。
继续检查高成本限制、零成本可分离性、标签互换与至少五个初值。
一般误差阈值为 1e-8，高成本跨区份额阈值为 1e-12，具体口径以 model.md 为准。
输出每项误差、阈值、通过状态和失败诊断；不调参或降低阈值迎合结果。
说明这些数值检查不能证明一般模型的唯一性或真实政策有效。
若材料缺失，请列明缺口；需要运行却无法运行时，区分拟执行步骤与已完成结果。
```

</div>
<div data-prompt-role="checks">

**拿到回复后看什么：** 检查是否依据方程重算，以及对称情形为何不产生空间偏向。

</div>
</div>
</details>



In [16]:
def relative(a, b):
    return float(np.max(np.abs(np.asarray(a)-np.asarray(b))/np.maximum(np.abs(b), 1.)))

def checks_for(s,p,name,checks):
    """用最终联合流量检查市场，不以候选租金公式自证出清。"""
    M=s['M']; rows=M.sum(1); cols=M.sum(0)
    def check(key,error,tol=1e-8):
        checks.append(dict(scenario=name,test=key,error=float(error),threshold=tol,passed=bool(error<=tol)))
    check('total_population',abs(M.sum()/p['N']-1))
    check('row_residents',relative(rows,s['R']))
    check('column_employment',relative(cols,s['E']))
    check('wage_marginal_product',relative(s['w'],p['beta']*p['Z']*cols**(p['beta']-1)))
    check('housing_clearing',relative(s['r']*p['H'],p['alpha']*(M@s['w'])))
    v=np.log(p['B'])[:,None]+np.log(s['w'])[None,:]-p['alpha']*np.log(s['r'])[:,None]-p['tau']
    check('joint_choice',relative(M,p['N']*softmax(p['theta']*v)))
    income=float((M@s['w']).sum())
    consumption=(1-p['alpha'])*income+float((s['r']*p['H']).sum())+(1-p['beta'])*s['Y'].sum()
    check('goods_resource',abs(consumption/s['Y'].sum()-1))
    check('log_ratio_residual',np.max(np.abs(s['residual'])))

In [17]:
# 核验用相对误差阈值记录结果，而不只读取求解器 success。
def check(name,error,tol=1e-8):
    checks.append(dict(scenario='special',test=name,error=float(error),threshold=tol,passed=bool(error<=tol)))
check('fixed_od_recomputed',relative(target['M'],M_input))
for key in ['M','R','E','w','r']: check('baseline_'+key,relative(b[key],target[key]))
expected={'H':[885169.7232957717,1497141.2594953028],
          'Z':[33464.6835987020,27311.6609829235], 'B':[1.,1.16855646176227]}
for key,val in expected.items(): check('calibration_'+key,relative(p[key],val))
for key in ['N','alpha','theta','beta','H','Z','B']: check('policy_fixed_'+key,relative(p[key],p1[key]))
zero=solve(changed(p,0),diagnostics=diagnostics,name='zero_shock')
for key in ['R','E','w','r','M']: check('zero_'+key,relative(zero[key],b[key]))

In [18]:
sym,st=calibrate([60000,60000],[9600,9600],[120,120])
sb=solve(sym,diagnostics=diagnostics,name='symmetric_baseline')
sp=changed(sym,.25); ss=solve(sp,diagnostics=diagnostics,name='symmetric_policy')
for key in ['R','E','w','r']: check('symmetric_'+key,relative(ss[key],st[key]))
check('symmetric_cross_increase',0. if ss['M'][0,1]>sb['M'][0,1] and ss['M'][1,0]>sb['M'][1,0] else 1.)
# 对称对照的小表：总量与价格不变，跨区份额提高。
symmetric_rows = [dict(情景=label, A居民=result['R'][0], A就业=result['E'][0],
                       A工资=result['w'][0], A房租=result['r'][0],
                       跨区通勤占比=100*summary(result,sb)['cross_share'])
                  for label,result in [('对称基准',sb),('对称政策',ss)]]
display(pd.DataFrame(symmetric_rows).style.format({k:'{:,.2f}' for k in symmetric_rows[0] if k!='情景'}))

,情景,A居民,A就业,A工资,A房租,跨区通勤占比
0,对称基准,"60,000.00","60,000.00","9,600.00",120.00,10.00
1,对称政策,"60,000.00","60,000.00","9,600.00",120.00,16.14


In [19]:
high=copy.deepcopy(p); high['tau']=np.array([[0.,20.],[20.,0.]])
hs=solve(high,diagnostics=diagnostics,name='tau20')
local=solve(p,diagnostics=diagnostics,name='local_only',restricted=True)
check('tau20_cross_share',summary(hs,b)['cross_share'],1e-12)
for key in ['R','E','w','r','M']: check('tau20_local_'+key,relative(hs[key],local[key]))
free=copy.deepcopy(p); free['tau']=np.zeros((2,2))
fs=solve(free,diagnostics=diagnostics,name='tau0')
check('tau0_separability',np.max(np.abs(fs['prob']-np.outer(fs['prob'].sum(1),fs['prob'].sum(0)))))
mirror=copy.deepcopy(p1)
for key in ['H','Z','B']: mirror[key]=mirror[key][::-1]
mirror['tau']=mirror['tau'][::-1,::-1]
ms=solve(mirror,diagnostics=diagnostics,name='labels_swapped')
for key in ['R','E','w','r']: check('mirror_'+key,relative(ms[key],s[key][::-1]))
check('mirror_M',relative(ms['M'],s['M'][::-1,::-1]))

In [20]:
for scenario,pars,base in [('baseline',p,b),('policy25',p1,s)]:
    for n,start in enumerate(STARTS):
        got=solve(pars,start,diagnostics,f'{scenario}_start{n}')
        check(f'{scenario}_start{n}',relative(got['M'],base['M']))
cases=[('baseline',p,b),('policy25',p1,s),('zero_shock',p,zero),
       ('symmetric_baseline',sym,sb),('symmetric_policy',sp,ss),('tau20',high,hs),('tau0',free,fs),('mirror',mirror,ms)]
for name,pars,result in cases: checks_for(result,pars,name,checks)

再看政策力度。固定主基准的参数，分别把双向跨区成本降低 0%、10%、25% 和 50%。表中的变化均相对于各自的无政策基准。

In [21]:
sensitivity=[]
for axis,grid in [('reduction',[0.,.1,.25,.5]),('theta',[2.,4.,6.]),('beta',[.8,.9,1.])]:
    for value in grid:
        theta=value if axis=='theta' else 4.
        beta=value if axis=='beta' else .9
        reduction=value if axis=='reduction' else .25
        cp,ct=calibrate(R,w,r,theta,beta,TAU0)
        cb=solve(cp,diagnostics=diagnostics,name=f'{axis}{value}_base')
        pp=changed(cp,reduction); cs=solve(pp,diagnostics=diagnostics,name=f'{axis}{value}_policy')
        for key in ['R','E','w','r','M']: check(f'{axis}{value}_baseline_{key}',relative(cb[key],ct[key]))
        checks_for(cb,cp,f'{axis}{value}_base',checks); checks_for(cs,pp,f'{axis}{value}_policy',checks)
        row=dict(axis=axis,value=value,theta=theta,beta=beta,reduction=reduction,tau0=TAU0,
                 baseline_cross_share=summary(cb,cb)['cross_share'],**summary(cs,cb),data_nature=NATURE)
        for key in ['R','E','w','r']:
            for j,region in enumerate(REGIONS): row[f'{key}_{region}_pct']=100*(cs[key][j]/cb[key][j]-1)
        sensitivity.append(row)

In [22]:
# 主表只展示政策力度，参数敏感性放在后面的补充表。
sensitivity_frame = pd.DataFrame(sensitivity)
def sensitivity_view(frame, label):
    out = frame[['value','R_B_pct','E_A_pct','welfare_pct']].copy()
    out.columns = [label,'B 居民变化 (%)','A 就业变化 (%)','福利增幅 (%)']
    return out
reduction_table = sensitivity_view(sensitivity_frame.query("axis == 'reduction'"), '成本降幅 (%)')
reduction_table['成本降幅 (%)'] *= 100
display(reduction_table.style.format('{:.2f}'))

,成本降幅 (%),B 居民变化 (%),A 就业变化 (%),福利增幅 (%)
0,0.00,0.00,0.00,0.00
1,10.00,0.76,2.26,0.74
2,25.00,2.04,6.00,2.14
3,50.00,4.51,12.82,5.50


在列出的正降幅情景中，B 居民增加与 A 就业集中都保留，降幅越大，变化幅度越大；零冲击返回基准。这是有限网格内的数值比较，不能外推为任意城市或任意参数下的定理。

改变行为或生产参数时，需要重新构造与该参数相容的基准。本章保持原始居民数、工资、租金和 $\tau_{0}$，每组先重建 OD，再反演基本面；政策计算内仍固定该组基本面。尤其是改变 $\theta$ 后，基准通勤份额会变化。这个约定与前章通过调整通勤成本匹配通勤份额的练习不同，不能混用。

<details class="agent-prompt" id="agent-m3-sensitivity" data-prompt-kind="exercise">
<summary><span class="agent-prompt-toggle" aria-hidden="true"></span><span>Agent · 讲义练习：改参数时，怎样重新建立基准</span></summary>
<div class="agent-prompt-body">
<div data-prompt-role="materials">

**准备材料：** 下载并交给 Agent：[模型说明](model.md) · [teaching-central-residential-baseline.csv](data/teaching-central-residential-baseline.csv)。 实现或复查时另提供已有程序、校准结果和本次输出；可用[教学 Notebook](transport.ipynb)作参考。

</div>
<div data-prompt-role="scope">

**适用条件：** 改变行为参数时保留原始居民和价格并重建 OD；不采用固定原 OD 的另一套校准规则。

</div>
<div data-prompt-role="task">

**练习任务**

```text
我提供了非对称两地区通勤成本政策模型的设定与此任务所需的输入材料。
请分别改变非对称两地区通勤政策模型的 theta 和 beta，按所提供模型的约定重新构造各组基准。
保持原始居民数、工资、租金和 tau0；每组先重建 OD，再反演 H、Z、B。
保持 tau0 不变，不通过调整通勤成本强行匹配原 OD。
每组政策内固定刚校准的基本面，按相同成本降幅求反事实。
实际运行后列出各组基准与政策变化，并保存市场残差。
参数网格是情景比较，不称为置信区间；政策力度变化与行为参数变化分别报告。
若材料缺失，请列明缺口；需要运行却无法运行时，区分拟执行步骤与已完成结果。
```

</div>
<div data-prompt-role="checks">

**拿到回复后看什么：** 重新校准发生在哪一步；是否误把 M2 的通勤匹配规则搬到本章。

</div>
</div>
</details>

<details class="agent-prompt" id="agent-m3-sensitivity-plan" data-prompt-kind="transfer">
<summary><span class="agent-prompt-toggle" aria-hidden="true"></span><span>Agent · 迁移到我的研究：改变参数时重建可比较的基准</span></summary>
<div class="agent-prompt-body">
<div data-prompt-role="materials">

**准备材料：** `{模型说明材料}`：方程与既有校准步骤；`{待比较参数}`：参数及取值范围；`{基准观测与来源}`：哪些数据真实可观测、哪些由模型生成；`{政策情景}`：每组共同实施的变化。

</div>
<div data-prompt-role="scope">

**适用条件：** 用于反事实模型的参数敏感性分析。保持原始观测还是生成的流量，需要结合模型决定；不能直接照搬另一个模型的校准顺序。

</div>
<div data-prompt-role="task">

**任务模板**

```text
我希望在 {模型说明材料} 下比较 {待比较参数}，基准信息为 {基准观测与来源}，共同政策为 {政策情景}。
请先提出每组参数的基准重建方案，区分固定的原始观测与重新生成或反演的量。
检查能否同时匹配这些目标；如约束冲突，列出原因与可选处理，不自行修改目标或通勤成本。
先解释步骤及依据，待方案确认后，再分别重建基准并施加相同政策。
每组政策内固定该组已校准的外生基本面，同时允许模型规定的内生量调整。
实际运行时保存每组基准、政策结果和经济残差，失败组也保留。
说明数值变化来自参数改变及相应校准；参数网格不称为置信区间，不调整参数迎合结论。
```

</div>
<div data-prompt-role="example">

**填写示例：** 以下是使用讲义材料的示范。先提供示例提到的材料，再单独复制；不要与上面的模板一起发送。

```text
我提供非对称两地区通勤模型说明，比较不同 theta 与 beta 下双向通勤成本下降 25% 的结果。
按该设定，固定原始居民数、工资、租金和基准通勤成本，先重建 OD，再反演住房、生产率和便利度。
请核对这一校准顺序后执行；各组政策内固定刚反演的基本面，保留市场残差。参数网格不解释为置信区间。
```

</div>
<div data-prompt-role="checks">

**拿到回复后看什么：** 哪些量被固定、哪些重新构造是否明确，各组政策与各自基准的比较是否一致。

</div>
</div>
</details>



In [23]:
# 参数表采用中文列名；网页读者可按需要展开。
for axis,label in [('theta','选择参数 θ'), ('beta','生产弹性 β')]:
    tab = sensitivity_view(sensitivity_frame[sensitivity_frame['axis']==axis], label)
    display(HTML('<details><summary>'+label+'：展开结果</summary>'
                 + tab.to_html(index=False, float_format=lambda x:f'{x:.2f}') + '</details>'))

选择参数 θ,B 居民变化 (%),A 就业变化 (%),福利增幅 (%)
2.00,1.00,2.58,4.08
4.00,2.04,6.00,2.14
6.00,2.23,7.97,1.09


生产弹性 β,B 居民变化 (%),A 就业变化 (%),福利增幅 (%)
0.80,2.36,4.86,2.15
0.90,2.04,6.00,2.14
1.00,1.53,7.84,2.11


在所列参数下，主结论的方向保持，但幅度有所变化。$\beta=1$ 时工资不再随就业变化，租金和选择仍会调整。这些网格没有统计置信区间的含义；若要评估真实交通项目，还需数据、参数识别，以及对拥堵、交通供给和建设成本等机制的处理。

## 6. 怎样运行和检查自己的结果

网页配套的 [Notebook](transport.ipynb) 将读取输入、构造通勤表、反演基本面、固定价格预测和均衡求解分开。按顺序运行，先核对基准，再读政策表。[离线练习包](https://lianxhcn.github.io/qsm101/articles/p06-transport-qsm/downloads/transport-workbook.zip) 包含 Notebook、原始输入、模型、参考脚本和配图，解压后可使用本地文件运行；Python 依赖需提前安装。

也可在仓库根目录运行参考脚本：

```text
python -X utf8 -m pip install -r articles/p06-transport-qsm/requirements.txt
python -X utf8 articles/p06-transport-qsm/code/qsm_transport_demo.py --output runs/p06-my-run
```

脚本只需 `requirements.txt` 中的计算依赖。交互运行 Notebook 还需 Jupyter 与 Python 内核，例如安装 `jupyterlab` 和 `ipykernel` 后打开文件。每次运行使用新输出目录，避免覆盖已有结果。

先检查 `verification.csv` 的 `passed` 列，再看 `equilibrium-results.csv` 的 `policy25` 行，最后用 `commuting-matrix-results.csv` 核对行列合计。求解器的状态与残差保存在 `solver-diagnostics.json`，即使状态标记不成功，也要结合最终残差和市场条件判断；不能隐藏该记录。参考流程共有 322 项核验，阈值及误差均随结果保存。

若提示数据不存在，应确认已解压完整练习包，或使用脚本的 `--data` 指向包含两份原始 CSV 的目录。若输出目录非空，换用新目录。若市场核验失败，先检查 OD 行列方向、住房支出中的工作地工资、政策内基本面是否保持不变，不要通过调参数让结果接近表中的数字。

In [24]:
def dump(path, obj):
    """用 UTF-8 保存可审计 JSON，不允许非标准 NaN。"""
    def convert(v):
        if isinstance(v, np.ndarray): return v.tolist()
        if isinstance(v, np.generic): return v.item()
        raise TypeError(str(type(v)))
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2,
                                   default=convert, allow_nan=False), encoding='utf-8')

def save_csv(path, rows):
    pd.DataFrame(rows).to_csv(path, index=False, encoding='utf-8-sig', float_format='%.17g')

In [25]:
equilibrium=[]; flows=[]; summaries=[]
for name,pars,result in cases[:7]:
    for i,reg in enumerate(REGIONS):
        equilibrium.append(dict(scenario=name,region=reg,**{k:result[k][i] for k in ['R','E','w','r']},data_nature=NATURE))
        for j,job in enumerate(REGIONS): flows.append(dict(scenario=name,home_region=reg,work_region=job,workers=result['M'][i,j],data_nature=NATURE))
    comparison_base=sb if name.startswith('symmetric') else b
    summaries.append(dict(scenario=name,**summary(result,comparison_base),data_nature=NATURE))
fixed_rows=[]
for name,result in [('fixed_prices_prediction',fixed),('general_equilibrium',s)]:
    for i,reg in enumerate(REGIONS):
        fixed_rows.append(dict(scenario=name,region=reg,**{k:result[k][i] for k in ['R','E','w','r']},**summary(result,b),data_nature=NATURE))

In [26]:
save_csv(output/'equilibrium-results.csv',equilibrium)
save_csv(output/'commuting-matrix-results.csv',flows)
save_csv(output/'commuting-summary.csv',summaries)
save_csv(output/'fixed-price-comparison.csv',fixed_rows)
save_csv(output/'sensitivity-results.csv',sensitivity)
save_csv(output/'verification.csv',checks)
save_csv(output/'teaching-central-residential-baseline.csv',[dict(region=reg,**{k:target[k][i] for k in ['R','E','w','r','H','Z','B']},data_nature=NATURE) for i,reg in enumerate(REGIONS)])
dump(output/'calibrated-parameters.json',dict(data_nature=NATURE,**p))
dump(output/'input-sha256.json',hashes)
dump(output/'solver-diagnostics.json',diagnostics)
dump(output/'main-results.json',dict(data_nature=NATURE,baseline=b,policy=s,fixed_prices=fixed,summary=summary(s,b)))
env=dict(python=sys.version,platform=platform.platform(),executable=sys.executable,
         packages={x:importlib.metadata.version(x) for x in ['numpy','scipy','pandas','matplotlib']},
         command=sys.argv,cwd=str(Path.cwd()),utc=datetime.now(timezone.utc).isoformat())
dump(output/'environment.json',env)

In [27]:
# 所有核验通过才完成；同时展示不能被 success 标记替代的残差信息。
failures = [item for item in checks if not item['passed']]
verification = dict(passed=not failures, count=len(checks), failed=failures,
                    max_error=max(item['error'] for item in checks), data_nature=NATURE)
dump(output/'verification.json', verification)
if failures:
    raise RuntimeError(f'{len(failures)} 项核验失败，请查 verification.csv')
status = pd.DataFrame(diagnostics)
display(pd.DataFrame([{'核验项目':len(checks), '通过项目':sum(item['passed'] for item in checks),
                       '最大误差':verification['max_error'], '求解记录':len(diagnostics),
                       'success 为 False 的记录':int((~status['success']).sum())}]))
print('输出已保存到本次运行的 runs/p06-notebook-时间戳 目录。')

,核验项目,通过项目,最大误差,求解记录,success 为 False 的记录
0,322,322,1.184436e-14,39,2


输出已保存到本次运行的 runs/p06-notebook-时间戳 目录。


可以用三个练习检查自己是否读懂了模型：

1. 把双向成本降幅设为零，解释为什么工资、租金、居民和就业都应返回基准。
2. 换回完全对称的基本面，核对双向通勤增加时，为什么两地总量仍不变。
3. 对比固定价格预测与一般均衡，分别解释 B 居民和 A 就业的差异，不预设两项差异必须同向。

若希望独立编程，可将[独立输入包](transport-agent-inputs.zip) 交给新目录中的新会话，只提供模型、原始数据和实现提示词，完成后再与参考结果比较。已完成的独立实现与四价格方程交叉核对见[参考结果说明](results/reference/README.md)。本章 Notebook 是参考流程的分步复现，不计作新的独立实现。

AI 可以辅助编程与排错，模型闭合、参数识别、福利对象和经济解释仍需研究者判断。本章的用途是理解通勤选择与两个市场如何相互影响；从教学算例走向真实政策，需要另行处理数据和模型适用性。

<details class="agent-prompt" id="agent-m3-adapt" data-prompt-kind="transfer">
<summary><span class="agent-prompt-toggle" aria-hidden="true"></span><span>Agent · 迁移到我的研究：把政策设想变成可核验的改动</span></summary>
<div class="agent-prompt-body">
<div data-prompt-role="materials">

**准备材料：** `{模型说明材料}`：现有设定和政策参数含义；`{已有程序}`：实现文件；`{原政策}`、`{新政策}`：两种情景与力度；`{结果目录}`：新结果保存位置。

</div>
<div data-prompt-role="scope">

**适用条件：** 适用于已有模型中的政策调整。若新政策改变成本的经济含义、预算约束或市场结构，先转入机制讨论。

</div>
<div data-prompt-role="task">

**任务模板**

```text
模型说明为 {模型说明材料}，程序为 {已有程序}。
我希望把 {原政策} 改为 {新政策}，结果保存到 {结果目录}。
请先判断这是否只涉及政策力度，列出改变的量、保持不变的量及模型依据。
如果它改变了行为机制、预算约束或市场结构，请先列出需要重写的方程与缺失数据，暂不改程序。
条件明确后指出代码修改位置，实施并实际运行；保留原情景作比较，不覆盖原结果。
根据模型核对零冲击、守恒和市场条件，分别报告求解状态与经济残差。
若比较固定价格与均衡结果，两者须共享相同基准与冲击；仅在模型支持时使用该分解。
解释主要传导机制和福利口径，不按预期方向调参。
材料不全或无法运行时明确说明，不把计划写成已实现结果。
```

</div>
<div data-prompt-role="example">

**填写示例：** 以下是使用讲义材料的示范。先提供示例提到的材料，再单独复制；不要与上面的模板一起发送。

```text
我提供非对称两地区通勤模型说明与程序，把双向跨区广义通勤成本下降 25% 改为下降 10%，结果写入 runs/cost-10pct。
成本仍解释为效用损失，保留原基准和行为参数。请先指出改动位置，再比较固定价格与一般均衡结果并检查残差。
如果我的要求变成从工资中扣除票价，请先讨论预算、交通收入与福利核算，不把它当成只改成本参数。
```

</div>
<div data-prompt-role="checks">

**拿到回复后看什么：** 政策力度与机制变化是否被区分，保留的固定量与福利对象是否符合模型。

</div>
</div>
</details>

[下载本章随读提示词](prompts/guided-prompts.md)。短卡片用于当前步骤；完整实现任务书用于从零实现练习。
